<a href="https://colab.research.google.com/github/wayhome/101colab/blob/main/Markov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gspread pandas numpy matplotlib scipy seaborn google-auth-oauthlib google-auth-httplib2

In [ ]:
import numpy as np

# Define states and transition matrix
states = ["Sunny", "Rainy"]
transition_matrix = np.array([
    [0.8, 0.2],  # From Sunny: 80% stay Sunny, 20% become Rainy
    [0.3, 0.7],  # From Rainy: 30% become Sunny, 70% stay Rainy
])

# Simulate the Markov Chain
def simulate_markov_chain(initial_state, transition_matrix, num_steps):
    current_state = initial_state
    sequence = [current_state]
    np.random.seed(42)  # For reproducibility
    for _ in range(num_steps):
        current_idx = states.index(current_state)
        next_idx = np.random.choice(len(states), p=transition_matrix[current_idx])
        current_state = states[next_idx]
        sequence.append(current_state)
    return sequence

# Run simulation: Start with "Sunny" for 7 days
weather_sequence = simulate_markov_chain("Sunny", transition_matrix, num_steps=7)
print("Weather Sequence:", weather_sequence)

Weather Sequence: ['Sunny', 'Sunny', 'Rainy', 'Rainy', 'Rainy', 'Sunny', 'Sunny', 'Sunny']


In [ ]:
import numpy as np
import pandas as pd
import gspread
from google.auth import default
import matplotlib.pyplot as plt
import seaborn as sns



try:
    from google.colab import auth
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# --- Google Sheet Loader Functions ---
gc = None

def authenticate_gsheets():
    global gc
    if gc: return gc
    try:
        if IS_COLAB: auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        print("Google Sheets authentication successful.")
        return gc
    except Exception as e:
        print(f"ERROR: Google Sheets Authentication Failed: {e}")
        return None

def load_data_from_gsheet(spreadsheet_name, worksheet_name):
    global gc
    if not gc:
        print("gspread client not initialized.")
        return None
    print(f"Attempting to load: Spreadsheet='{spreadsheet_name}', Worksheet='{worksheet_name}'")
    try:
        spreadsheet = gc.open(spreadsheet_name)
        worksheet = spreadsheet.worksheet(worksheet_name)
        data = worksheet.get_all_values()
        if not data or len(data) < 2:
            print(f"Warning: Worksheet '{worksheet_name}' is empty or has no data rows.")
            return None
        headers = data[0]
        df = pd.DataFrame(data[1:], columns=headers)

        if 'Date' not in df.columns:
            print(f"ERROR: 'Date' column not found in '{worksheet_name}'.")
            return None
        formats_to_try = ['%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%m/%d/%Y', '%d/%m/%Y']
        parsed_date = False
        original_dates = df['Date'].copy()
        for fmt in formats_to_try:
            try:
                df['Date'] = pd.to_datetime(original_dates, format=fmt, errors='coerce')
                if not df['Date'].isnull().all():
                    parsed_date = True
                    break
            except (ValueError, TypeError): continue
        if not parsed_date: df['Date'] = pd.to_datetime(original_dates, errors='coerce')
        df.dropna(subset=['Date'], inplace=True)
        if not df.empty:
            df.set_index('Date', inplace=True)
            df.sort_index(inplace=True)
        else: return None

        required_col = 'Close'
        if required_col not in df.columns:
            print(f"ERROR: Required column '{required_col}' not found.")
            return None
        df[required_col] = df[required_col].replace({'-': np.nan, '': np.nan, ' ': np.nan, '#N/A': np.nan})
        df[required_col] = pd.to_numeric(df[required_col], errors='coerce')
        df.dropna(subset=[required_col], inplace=True)

        for col in ['Open', 'High', 'Low', 'Volume']:
            if col in df.columns:
                df[col] = df[col].replace({'-': np.nan, '': np.nan, ' ': np.nan, '#N/A': np.nan})
                df[col] = pd.to_numeric(df[col], errors='coerce').ffill()
        if df.empty:
            print(f"Warning: DataFrame for '{worksheet_name}' became empty after preprocessing.")
            return None
        print(f"Successfully loaded and preprocessed '{worksheet_name}' ({len(df)} rows).")
        return df
    except Exception as e:
        print(f"ERROR loading '{worksheet_name}': {str(e)}")
        return None

# --- Markov Chain Functions ---
def create_states(prices, n_states=5):
    if prices.empty or len(prices) < 2: return pd.Series(dtype=int)
    percent_change = prices.pct_change().dropna()
    if percent_change.empty: return pd.Series(dtype=int)
    percent_change_values = percent_change.values
    try:
        # Ensure enough unique values for qcut, otherwise use cut
        if len(np.unique(percent_change_values)) >= n_states:
             states_cat = pd.qcut(percent_change_values, q=n_states, labels=False, duplicates='drop') # Use labels=False for direct int output
        else:
             states_cat = pd.cut(percent_change_values, bins=n_states, labels=False, retbins=False, duplicates='drop')
    except ValueError: # Fallback if qcut/cut fails (e.g. not enough unique points or bins issue)
        print(f"Warning: Could not discretize states using cut or qcut for {n_states} states. Trying with fewer bins if possible or failing.")
        # Try with a reduced number of states if n_states is too high for the data variation
        try:
            num_unique_pct = len(np.unique(percent_change_values))
            bins_to_try = min(n_states, num_unique_pct) if num_unique_pct > 0 else 1
            if bins_to_try < 1: bins_to_try = 1 # Must have at least 1 bin

            if bins_to_try == 1 and num_unique_pct > 0 : # if only one bin possible, all go to state 0
                 states_cat = np.zeros(len(percent_change_values), dtype=int)
            elif bins_to_try > 1 :
                 states_cat = pd.cut(percent_change_values, bins=bins_to_try, labels=False, retbins=False, duplicates='drop')
                 # If states_cat has fewer than n_states categories, it's okay, matrix will adapt.
            else: # num_unique_pct == 0 (e.g. all prices were same, pct_change is all 0 or empty after dropna)
                 print("Warning: No price variation to create states.")
                 return pd.Series(dtype=int) # No states can be formed
        except Exception as e_fallback:
            print(f"ERROR: Fallback state creation failed: {e_fallback}")
            return pd.Series(dtype=int)

    # states_cat might contain NaNs if some values fall outside all bins (though labels=False should map to -1 or similar)
    # Ensure states are non-negative integers. pd.cut with labels=False gives 0-indexed labels or -1 for NaNs.
    # We want to ensure states are in range [0, n_labels-1]
    # If states_cat is already integer array from numpy zeros, it's fine.
    if isinstance(states_cat, pd.Series): states_cat = states_cat.values # Ensure numpy array

    # Handle any potential NaNs or -1s from cut/qcut if values were outside explicit bins
    # This step ensures all are valid indices for the transition matrix
    # Create a series and map -1 (or NaN) to a specific state or drop, here we ensure integer type
    # and rely on build_transition_matrix to handle if states are outside expected range due to reduced bins.
    # However, labels=False from pd.cut/qcut should give 0 to k-1 labels.
    valid_states = states_cat[~np.isnan(states_cat) & (states_cat >= 0)].astype(int)
    valid_indices = percent_change.index[~np.isnan(states_cat) & (states_cat >= 0)]

    return pd.Series(valid_states, index=valid_indices)


def build_transition_matrix(states_series, n_states=5):
    # states_series might have fewer than n_states unique values if create_states adapted.
    # The matrix should still be n_states x n_states.
    # Max observed state will determine actual used dimension for counting.
    if states_series.empty or len(states_series) < 2:
        return np.full((n_states, n_states), np.nan)

    # Determine the actual number of states present in the data, capped by n_states
    # This handles if create_states returned fewer than n_states categories.
    # Example: if n_states=5 but data only shows 3 states (0,1,2), matrix is 5x5 but only 0,1,2 rows/cols get counts.

    matrix = np.zeros((n_states, n_states))
    for (i, j) in zip(states_series[:-1], states_series[1:]):
        # Ensure i and j are valid integers and within the desired n_states bounds
        if pd.notna(i) and pd.notna(j):
            i_int, j_int = int(i), int(j)
            if 0 <= i_int < n_states and 0 <= j_int < n_states:
                matrix[i_int][j_int] += 1
            # Else: state observed is outside the expected range [0, n_states-1], ignore.
            # This might happen if create_states dynamically created more states than n_states,
            # which it shouldn't with labels=False and bins=n_states.

    row_sums = matrix.sum(axis=1, keepdims=True)
    # For rows with sum 0 (state never occurred as 'from' state, or never transitioned), fill with NaN.
    return np.divide(matrix, row_sums, out=np.full_like(matrix, np.nan), where=(row_sums != 0))

def plot_transition_matrix(matrix, labels):
    """Create a heatmap of the transition matrix"""
    plt.figure(figsize=(10, 8))
    # Mask NaNs for plotting or they'll cause issues with annotations/colormap
    # sns.heatmap handles NaNs by not coloring those cells by default (depending on version/settings)
    # but fmt might error if it encounters NaN.
    # We can plot np.nan_to_num(matrix, nan=-1) and use a colormap that highlights -1,
    # or just let heatmap handle it. Default is usually fine.
    sns.heatmap(matrix, annot=True, cmap="YlGnBu", fmt=".2f",
                xticklabels=labels, yticklabels=labels, cbar=True, vmin=0, vmax=1) # Ensure colorbar is sensible
    plt.title("Transition Probability Matrix")
    plt.xlabel("Next State")
    plt.ylabel("Current State")
    plt.tight_layout()
    plt.show()

def compute_steady_state(trans_matrix, iterations=200):
    """Compute the steady state by matrix power iteration, ensuring matrix is stochastic."""
    P_work = trans_matrix.copy()
    n_states = P_work.shape[0]

    if n_states == 0:
        print("Warning: Transition matrix is empty. Cannot compute steady state.")
        return np.array([])

    # Identify rows that were all NaN in the original trans_matrix.
    # These represent states for which no outgoing transitions were observed.
    all_nan_rows_original = np.all(np.isnan(P_work), axis=1)

    # Convert all NaNs to 0.0. This handles isolated NaNs if any.
    P_work = np.nan_to_num(P_work, nan=0.0)

    # Ensure P_work is row-stochastic.
    for i in range(n_states):
        if all_nan_rows_original[i]:
            # If state 'i' had no observed outgoing transitions (row was all NaN),
            # assume it's an absorbing state (P[i,i] = 1). This is a common convention.
            P_work[i, :] = 0.0
            P_work[i, i] = 1.0
        else:
            # For rows that were not all NaN originally:
            # Normalize them to sum to 1 if they don't already.
            # This can happen if nan_to_num changed some entries to 0, affecting the sum,
            # or if build_transition_matrix had precision issues or a state had no valid next states observed.
            row_sum = np.sum(P_work[i, :])
            if row_sum > 1e-9: # If row is not effectively all zeros
                P_work[i, :] /= row_sum
            else:
                # If row sum is zero (e.g., state observed as 'from' but all 'to' were NaN states, or no 'to' states at all)
                # also make it absorbing as a fallback strategy to ensure stochasticity.
                P_work[i, :] = 0.0
                P_work[i, i] = 1.0

    # At this point, P_work should be a row-stochastic matrix.
    # Use numpy.linalg.matrix_power to compute P_work^iterations.
    # If P_work is stochastic, P_work^iterations will also be stochastic.
    try:
        # High power of a stochastic matrix; rows should converge for an ergodic chain.
        P_final = np.linalg.matrix_power(P_work, iterations)
    except np.linalg.LinAlgError as e:
        print(f"Error during matrix_power: {e}. Returning NaN steady state.")
        return np.full(n_states, np.nan)

    # For a regular (ergodic) Markov chain, all rows of P_final converge to the same steady-state vector.
    # We can take the first row. Its sum should be 1.0.
    steady_state_vector = P_final[0, :]

    # Verify sum and non-negativity (should hold if P_work was properly stochastic)
    if not (np.isclose(np.sum(steady_state_vector), 1.0) and np.all(steady_state_vector >= -1e-9)):
        print(f"Warning: Steady-state vector from P_final[0,:] (sum={np.sum(steady_state_vector)}) is problematic. Checking other rows or returning NaN.")
        # Try to find a "good" row if the first one is bad (e.g. due to complex reducibility)
        for r_idx in range(1, n_states):
            row_vec = P_final[r_idx, :]
            if np.isclose(np.sum(row_vec), 1.0) and np.all(row_vec >= -1e-9):
                print(f"Using row {r_idx} from P_final for steady state.")
                return np.maximum(0, row_vec) # Clean up tiny negatives

        # If no row is good, indicate failure.
        print("Could not find a valid steady-state vector in P_final rows.")
        return np.full(n_states, np.nan)

    return np.maximum(0, steady_state_vector) # Clean up tiny negatives from float arithmetic


# --- Main execution ---
if __name__ == "__main__":
    print("--- Markov Chain Transition Matrix Generator with Current State ---")

    if not authenticate_gsheets():
        print("Exiting due to authentication failure.")
        exit()

    # Configuration
    spreadsheet_file_name = 'AS'
    worksheet_ticker = 'AMCR'
    start_date_str = "2020-01-01"
    end_date_str = "2025-05-07"
    n_markov_states = 5 # This is the target number of states for create_states and matrix dimensions
    state_labels = ["Big Drop", "Small Drop", "Neutral", "Small Rise", "Big Rise"]
    if len(state_labels) != n_markov_states:
        print(f"Warning: Mismatch between n_markov_states ({n_markov_states}) and length of state_labels ({len(state_labels)}). Adjusting labels or n_states.")
        # Adjust n_markov_states to match labels, or truncate/extend labels. Here, we'll assume labels define n_states.
        n_markov_states = len(state_labels)


    # Load and filter data
    data = load_data_from_gsheet(spreadsheet_file_name, worksheet_ticker)
    if data is None or data.empty:
        print("Failed to load data or data is empty. Exiting.")
        exit()

    if start_date_str: data = data[data.index >= pd.to_datetime(start_date_str)]
    if end_date_str: data = data[data.index <= pd.to_datetime(end_date_str)]
    if data.empty:
        print("DataFrame became empty after date filtering. Exiting.")
        exit()

    prices = data['Close']
    if len(prices) < 2: # Need at least 2 prices for one pct_change, 3 prices for two states for one transition
        print(f"Not enough price data points ({len(prices)}) after filtering. Need at least 3 for one transition. Exiting.")
        exit()

    # Create states
    # n_markov_states is passed to create_states.
    states_series = create_states(prices, n_states=n_markov_states)
    if states_series.empty or len(states_series) < 2 : # Need at least 2 states in sequence for one transition
        print("Could not create states or not enough states in sequence for transition matrix. Exiting.")
        exit()

    # Build transition matrix, ensuring it's n_markov_states x n_markov_states
    trans_matrix = build_transition_matrix(states_series, n_states=n_markov_states)
    if np.all(np.isnan(trans_matrix)): # Check if the entire matrix is NaN
        print("Failed to build a valid transition matrix (all NaN). Exiting.")
        exit()

    # Get current state (most recent)
    current_state_idx = states_series.iloc[-1]
    # Ensure current_state_idx is within bounds for state_labels
    if not (0 <= current_state_idx < len(state_labels)):
        print(f"Error: Current state index {current_state_idx} is out of bounds for state_labels (len {len(state_labels)}).")
        # This can happen if create_states produced state indices outside [0, n_markov_states-1]
        # or if n_markov_states changed after create_states.
        # The current create_states and build_transition_matrix are designed for [0, n_states-1].
        exit()
    current_state_label = state_labels[current_state_idx]
    current_date = states_series.index[-1].strftime('%Y-%m-%d')

    current_price = prices.loc[states_series.index[-1]]

    if len(states_series.index) >= 2:
        prev_state_date = states_series.index[-2]
        if prev_state_date in prices.index and states_series.index[-1] in prices.index:
            price_today = prices.loc[states_series.index[-1]]
            price_yesterday = prices.loc[prev_state_date]
            if price_yesterday != 0:
                 price_change_pct = (price_today - price_yesterday) / price_yesterday * 100
            else: price_change_pct = np.nan
        else: price_change_pct = np.nan
    else: price_change_pct = np.nan

    print(f"\nCurrent Market State (as of {current_date}):")
    print(f"- Price: ${current_price:.2f}")
    if not np.isnan(price_change_pct):
        print(f"- Daily Change leading to this state: {price_change_pct:+.2f}%")
    else:
        print("- Daily Change leading to this state: N/A")
    print(f"- State: {current_state_label} (Index: {current_state_idx})")

    print("\nTransition Matrix (Probabilities for Next State):")
    trans_matrix_df = pd.DataFrame(
        trans_matrix,
        index=state_labels[:n_markov_states], # Ensure labels match matrix dim
        columns=[f"Next {s}" for s in state_labels[:n_markov_states]]
    )
    print(trans_matrix_df.round(4).to_string())

    print(f"\nNext State Probabilities (Given Current = {current_state_label}):")
    if 0 <= current_state_idx < trans_matrix.shape[0]:
        current_probs = trans_matrix[current_state_idx]
        if np.all(np.isnan(current_probs)):
            print(f"- Probabilities from state {current_state_label} are undefined (state likely not visited as origin, or no outgoing transitions observed).")
        else:
            for state_lbl, prob in zip(state_labels[:n_markov_states], current_probs):
                if pd.notna(prob): print(f"- {state_lbl:<12}: {prob:.2%}")
                else: print(f"- {state_lbl:<12}: N/A")
    else:
        print(f"- Current state index {current_state_idx} is out of bounds for the transition matrix.")

    if not np.all(np.isnan(trans_matrix)):
        steady_state_probs = compute_steady_state(trans_matrix.copy())
        if steady_state_probs.size > 0 and not np.all(np.isnan(steady_state_probs)):
            print("\nLong-term Market State Distribution (Steady State):")
            for state_lbl, prob in zip(state_labels[:n_markov_states], steady_state_probs):
                if pd.notna(prob): print(f"- {state_lbl:<12}: {prob:.2%}")
                else: print(f"- {state_lbl:<12}: N/A")
        else:
            print("\nCould not compute a valid steady-state distribution.")
    else:
        print("\nSkipping steady-state calculation due to invalid transition matrix.")


    print("\n--- Analysis Complete ---")

    if not np.all(np.isnan(trans_matrix)) and state_labels:
        plot_transition_matrix(trans_matrix, state_labels[:n_markov_states])
    else:
        print("Skipping plot: Transition matrix or labels are not available/valid.")

--- Markov Chain Transition Matrix Generator with Current State ---
